Do laboratório à equação diferencial: uma sensibilização sobre significados

In [10]:
from __future__ import annotations

from collections.abc import Callable
from dataclasses import dataclass, field
from typing import Any

import ipywidgets as widgets


type Listener = Callable[[Any], None]
type Transform = Callable[..., Any]


def identity(x: Any) -> Any:
    """Retorna o argumento sem modificações."""
    return x


@dataclass(slots=True)
class ReactiveNode:
    """
    Nó reativo que armazena um valor e notifica assinantes quando esse valor muda.

    Esta classe é a base do pequeno sistema de dependências. Um nó pode ser:

    - uma fonte primária, como o valor de um widget;
    - um valor derivado de outros nós;
    - um produtor de valores para widgets de destino.

    Parameters
    ----------
    value
        Valor inicial do nó.
    listeners
        Lista de callbacks registrados para reagir a mudanças de valor.
    """

    value: Any = None
    listeners: list[Listener] = field(default_factory=list)

    def get(self) -> Any:
        """
        Retorna o valor atual do nó.

        Returns
        -------
        Any
            Valor atualmente armazenado.
        """
        return self.value

    def set(self, value: Any) -> None:
        """
        Atualiza o valor do nó e notifica os assinantes.

        Parameters
        ----------
        value
            Novo valor do nó.
        """
        self.value = value
        self._notify()

    def subscribe(
        self,
        listener: Listener,
        *,
        call_immediately: bool = False,
    ) -> ReactiveNode:
        """
        Registra um callback para reagir às mudanças deste nó.

        Parameters
        ----------
        listener
            Função que recebe o novo valor do nó.
        call_immediately
            Se ``True``, o callback é chamado imediatamente com o valor atual.

        Returns
        -------
        ReactiveNode
            O próprio nó, permitindo encadeamento.
        """
        self.listeners.append(listener)
        if call_immediately:
            listener(self.value)
        return self

    def unsubscribe(self, listener: Listener) -> None:
        """
        Remove um callback previamente registrado.

        Parameters
        ----------
        listener
            Callback a ser removido.
        """
        if listener in self.listeners:
            self.listeners.remove(listener)

    def _notify(self) -> None:
        """
        Notifica todos os callbacks registrados com o valor atual.
        """
        for listener in list(self.listeners):
            listener(self.value)

    def map(
        self,
        func: Callable[[Any], Any],
        *,
        initialize: bool = True,
    ) -> ReactiveNode:
        """
        Cria um novo nó derivado deste, aplicando uma transformação ao seu valor.

        Parameters
        ----------
        func
            Função de transformação unária.
        initialize
            Se ``True``, o nó derivado já nasce com base no valor atual.

        Returns
        -------
        ReactiveNode
            Novo nó derivado.
        """
        initial = func(self.value) if initialize else None
        child = ReactiveNode(initial)

        def listener(value: Any) -> None:
            child.set(func(value))

        self.subscribe(listener)
        return child

    def to_widget(
        self,
        widget: widgets.Widget,
        attr: str,
        transform: Callable[[Any], Any] = identity,
        *,
        initialize: bool = True,
    ) -> ReactiveNode:
        """
        Liga este nó a um atributo de um widget de destino.

        Sempre que o valor do nó mudar, o atributo do widget será atualizado.

        Parameters
        ----------
        widget
            Widget de destino.
        attr
            Nome do atributo a ser atualizado.
        transform
            Transformação aplicada ao valor do nó antes da atribuição.
        initialize
            Se ``True``, o widget é inicializado imediatamente.

        Returns
        -------
        ReactiveNode
            O próprio nó, permitindo encadeamento.
        """
        def listener(value: Any) -> None:
            setattr(widget, attr, transform(value))

        self.subscribe(listener, call_immediately=initialize)
        return self


class WidgetNode(ReactiveNode):
    """
    Nó reativo cuja fonte é um atributo observável de um widget do ipywidgets.

    Parameters
    ----------
    widget
        Widget de origem.
    attr
        Nome do atributo observado, por exemplo ``"value"``.
    """

    def __init__(self, widget: widgets.Widget, attr: str) -> None:
        self.widget = widget
        self.attr = attr
        self._widget_handler = self._make_handler()
        super().__init__(getattr(widget, attr))
        widget.observe(self._widget_handler, names=attr)

    def _make_handler(self) -> Callable[[dict[str, Any]], None]:
        """
        Cria o callback que recebe eventos do ipywidgets.observe.

        Returns
        -------
        Callable[[dict[str, Any]], None]
            Função de callback.
        """
        def handler(change: dict[str, Any]) -> None:
            self.set(change["new"])

        return handler

    def unlink(self) -> None:
        """
        Remove a observação do widget de origem.
        """
        self.widget.unobserve(self._widget_handler, names=self.attr)


def bind(widget: widgets.Widget, attr: str = "value") -> WidgetNode:
    """
    Cria um nó reativo a partir de um atributo de um widget.

    Parameters
    ----------
    widget
        Widget de origem.
    attr
        Atributo observado.

    Returns
    -------
    WidgetNode
        Nó reativo vinculado ao widget.
    """
    return WidgetNode(widget, attr)


def computed(
    *nodes: ReactiveNode,
    func: Transform,
    initialize: bool = True,
) -> ReactiveNode:
    """
    Cria um nó derivado de um ou mais nós de entrada.

    Sempre que qualquer dependência mudar, o valor é recalculado aplicando
    ``func`` aos valores atuais de todas as dependências.

    Parameters
    ----------
    *nodes
        Nós de entrada.
    func
        Função que combina os valores atuais das dependências.
    initialize
        Se ``True``, o nó derivado é inicializado imediatamente.

    Returns
    -------
    ReactiveNode
        Novo nó derivado.
    """
    def evaluate() -> Any:
        return func(*(node.get() for node in nodes))

    result = ReactiveNode(evaluate() if initialize else None)

    def recompute(_: Any) -> None:
        result.set(evaluate())

    for node in nodes:
        node.subscribe(recompute)

    return result

In [11]:
import ipywidgets as widgets
from IPython.display import display

xslider = widgets.FloatSlider(min=0, max=100, step=0.1, value=50, description="x")
yprogress = widgets.FloatProgress(min=0, max=10000, description="x²")

x = bind(xslider, "value")
x_squared = x.map(lambda v: v**2)

x_squared.to_widget(yprogress, "value")

display(xslider)
display(yprogress)

FloatSlider(value=50.0, description='x')

FloatProgress(value=2500.0, description='x²', max=10000.0)